# Step-7a: Histogram-Matched Subset GENERATION (Dick 2021 Pool)

Standalone subset-selection pre-process, split out of the step-7
training notebook. Minimizes the distance between candidate-subset
and full-pool histograms over $(\rho^{1/3}, s, \alpha)$ and writes a
`subset_index_log.json` ledger + per-spec `subset.traj` files that the
step-7 training notebook (`gga_training_example-step7.ipynb`) and the
SLURM harness CONSUME read-only.

**Critical:** $\alpha$ enters the subset-selection objective only -- the
trained GGA network does NOT consume it (C4-03: toggle with the
`STEP7_IGNORE_ALPHA` env var). Reference: Dick & Fernandez-Serra,
*Phys. Rev. B* **104**, L161109 (2021), SI II.


In [ ]:
import gc
import json
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

import jax
# JAX config (mirrors step-6 cell 2): pin x64 dtype and CPU device
# *before* importing jnp or any library that may trigger JAX tracing.
# These must not change later in the notebook -- flipping
# jax_enable_x64 after traces are cached produces inconsistent dtypes
# and breaks numerical comparisons against step-5/6 checkpoints.
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_default_device", jax.devices("cpu")[0])
# Persistent compilation cache: writes compiled XLA HLO/LLVM to disk
# so that kernel restarts (e.g. after a crash) and across-spec calls
# in the 80-spec grid don't re-pay the full compile cost.
os.makedirs(".jax_compilation_cache", exist_ok=True)
jax.config.update("jax_compilation_cache_dir", ".jax_compilation_cache")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 1.0)
import jax.numpy as jnp
import equinox as eqx

# tqdm.auto picks tqdm.notebook.tqdm (ipywidgets) under JupyterLab
# and tqdm.std.tqdm in a plain script/terminal, so the same symbol
# gives a sensible progress bar in either context. Imported globally
# (step-6 parity) so per-task / per-spec / per-step bars in later
# cells share one library instance and one display.
from tqdm.auto import tqdm

# BLAS thread cap: PySCF SCF (used by descriptor extraction, eval,
# and run_test) is CPU-bound via OpenMP-threaded BLAS. Because this
# kernel ALSO imports JAX (which maintains its own thread pool),
# leaving PySCF at its default thread count (N-cores) causes
# oversubscription. Mirrors step-6 cell 2.
from contextlib import contextmanager as _contextmanager
from pyscf import lib as _pyscf_lib

@_contextmanager
def _capped_blas_threads(n=4):
    """Temporarily cap PySCF's BLAS thread count to `n`."""
    _prev = _pyscf_lib.num_threads()
    _pyscf_lib.num_threads(min(_prev, int(n)))
    try:
        yield
    finally:
        _pyscf_lib.num_threads(_prev)

import xcquinox.alec as alec
from xcquinox.alec import subset_selection as ss
from xcquinox.alec import dfs_pool
from xcquinox.alec import losses

REPO = Path('/home/awills/Documents/Research/xcquinox')
# C4-03: alpha (the meta-GGA tau descriptor) enters subset SELECTION
# only -- the GGA network is blind to it. Set STEP7_IGNORE_ALPHA=1 to
# drop alpha from the histogram-matching objective
# (descriptor_weights={'alpha': 0.0}); default keeps the alpha-weighted
# selection. Each mode writes to its OWN root so the alpha-on and
# alpha-off experiments (x L2/JSD metrics) coexist without clobbering.
_IGNORE_ALPHA = os.environ.get('STEP7_IGNORE_ALPHA', '0') == '1'
DESCRIPTOR_WEIGHTS = {'alpha': 0.0} if _IGNORE_ALPHA else None
_ALPHA_MODE = 'alpha_off' if _IGNORE_ALPHA else 'alpha_on'
STEP7_ROOT = REPO / 'notebooks' / 'checkpoints_step7' / _ALPHA_MODE
print(f'[C4-03] subset-selection descriptor mode={_ALPHA_MODE} '
      f'descriptor_weights={DESCRIPTOR_WEIGHTS}')
DESCRIPTOR_CACHE = STEP7_ROOT / 'subset_descriptors'
REF_HIST_CACHE = STEP7_ROOT / 'dfs_pool_full_hist'
DESCRIPTOR_CACHE.mkdir(parents=True, exist_ok=True)
REF_HIST_CACHE.mkdir(parents=True, exist_ok=True)
EXTERNAL_REFS_DIR = STEP7_ROOT / 'external_refs'
EXTERNAL_REFS_DIR.mkdir(parents=True, exist_ok=True)
DISTRIBUTIONS_DIR = STEP7_ROOT / 'subset_index_log_distributions'
DISTRIBUTIONS_DIR.mkdir(parents=True, exist_ok=True)

# Step-6 parity: BASIS / GRID_LEVEL match step-5/6 conventions so
# checkpoints carry across notebooks and descriptors are commensurate.
BASIS = 'def2-svp'
GRID_LEVEL = 1

pool = dfs_pool.build_dfs_pool()
print(f'JAX x64: {jax.config.read("jax_enable_x64")}; '
      f'default device: {jax.config.jax_default_device}')
print(f'Dick 2021 SI II training pool: {pool["n_total"]} entries')
print(f'  AE molecules: {len(pool["ae_molecules"])}')
print(f'  BH76 reactions: {len(pool["bh76_reactions"])}')
print(f'  IP13 pairs: {len(pool["ip13_pairs"])}')
print(f'  Atom refs: {len(pool["atom_refs"])}')


## 1. Mixed-Pool Descriptor Extraction (cached)

Build the 26-point Dick training pool (21 AE + 3 BH76 + 2 IP13) as
`TrainingPoint` records (each carries the species needed by its
loss channel, with atom anchors restricted to the Dick set
$\{\mathrm{H}, \mathrm{Li}\}$). Run a single PBE SCF
at def2-svp / grid_level=1 per UNIQUE species (deduped by
$(name, charge, spin)$) and cache descriptors as
`subset_descriptors/<name>_c<charge>_s<spin>.npz`. Then
concatenate per-species descriptors across each TrainingPoint's
species (design choice "a") to obtain one descriptor block
per point — multi-species points (reactions) thus weigh more
in the reference distribution proportional to grid-point count.


In [ ]:
from xcquinox.alec.training_points import (
    build_dfs_pool_points, species_union_from_points,
    DICK_ATOM_REGULARIZER_SYMS,
)
points = build_dfs_pool_points()
_by_kind = {'ae': 0, 'bh76': 0, 'ip13': 0}
for p in points:
    _by_kind[p.kind] += 1
print(f'Mixed pool: {len(points)} points '
      f'({_by_kind["ae"]} AE + {_by_kind["bh76"]} BH76 + '
      f'{_by_kind["ip13"]} IP13). Dick atom regularizer set: '
      f'{DICK_ATOM_REGULARIZER_SYMS}.')
_all_species = species_union_from_points(points)
print(f'Unique species across pool (dedup by name+charge+spin): '
      f'{len(_all_species)}')
species_descriptors = ss.extract_descriptors_for_species(
    _all_species, basis=BASIS, grid_level=GRID_LEVEL,
    cache_dir=DESCRIPTOR_CACHE,
)
for idx, p in enumerate(points):
    print(f'  {idx:2d} [{p.kind:4s}] {p.name:24s} '
          f'n_species={len(p.species)}')
point_descriptors = ss.concatenate_point_descriptors(
    points, species_descriptors)
print(f'Built per-point descriptors: {len(point_descriptors)} '
      f'(parallel to points)')


## 2. Reference-Histogram Builder

Concatenate descriptors across the full 26-point mixed pool
(per-point blocks already concatenated across each point's
species) and build 3 200-bin log10 density-normalized histograms
over $(\rho^{1/3}, s, \alpha)$. Same edges are reused for
every candidate-subset histogram.


In [ ]:
import numpy as np
h_ref, edges = ss.build_reference_histograms(point_descriptors)
ref_path = REF_HIST_CACHE / 'reference.npz'
np.savez(ref_path,
         h_ref_rho=h_ref['rho_third'], e_rho=edges['rho_third'],
         h_ref_s=h_ref['s'],         e_s=edges['s'],
         h_ref_alpha=h_ref['alpha'], e_alpha=edges['alpha'])
print(f'Wrote reference histograms to {ref_path}')
for k in ('rho_third', 's', 'alpha'):
    print(f'  {k:10s} histogram: shape={h_ref[k].shape}, sum={h_ref[k].sum():.4f}')


## 3. Subset Generation Sweep (mixed pool)

For each $(r, \text{metric})$, call `select_subset` over the
26-point mixed pool, materialize the chosen-points'
deduplicated species union as `subset.traj`, and record the
chosen pool indices + metric value to a JSON ledger. Atom
anchors are restricted to the Dick set $\{\mathrm{H}, \mathrm{Li}\}$
by `TrainingPoint` construction — nothing is forcibly added.
Spec count = $|\text{sizes}| \times |\text{metrics}| \times |\text{solvers}|$.


In [ ]:
from ase.io import write as ase_write
import json
import os as _os_smoke

_SMOKE = _os_smoke.environ.get('STEP7_SMOKE_ONLY', '0') == '1'
if _SMOKE:
    SUBSET_SIZES = (2,)
    METRICS = ('l2',)
    SOLVERS = ('oneshot',)
    TRAIN_N_STEPS = 5
    print('[SMOKE MODE] restricted to 1 spec: l2/r=2/oneshot, 5 steps')
else:
    SUBSET_SIZES = (1, 2, 3, 4, 5, 6, 7, 12, 15, 18)
    METRICS = ('l2', 'jsd')
    SOLVERS = ('oneshot', 'full_3')
    TRAIN_N_STEPS = 100
LR_START = 0.01
LR_END = 1e-05
LR_DECAY_START = 0.2
GRAD_CLIP = 1.0
ARCH_NAME = 'deep_combined_attn'
LOSS_NAME = 'L5_gradnorm_vxc_step7'

import time as _time
import math as _math
ledger_path = STEP7_ROOT / 'subset_index_log.json'
# Each (metric, r) entry stores: chosen_indices into the 26-point
# mixed pool, the per-point kinds + names (for inspection), and
# the metric value, so a killed run resumes at the next un-
# enumerated pair without re-running the L2/JSD search for sizes
# already on disk.
subset_index_log: dict = {}
if ledger_path.exists():
    _existing = json.loads(ledger_path.read_text())
    for _slashkey, _val in _existing.items():
        _m, _r = _slashkey.split('/')
        subset_index_log[(_m, int(_r))] = _val
    print(f'Resumed ledger: {len(subset_index_log)} (metric, r) '
          f'entries already on disk at {ledger_path}')

def _write_ledger():
    """Serialize subset_index_log to disk.  Called after EACH (metric, r)
    pair completes so partial progress survives a Ctrl-C / OOM / crash."""
    ledger_path.write_text(json.dumps(
        {f'{k[0]}/{k[1]}': v for k, v in subset_index_log.items()},
        indent=2))

_n_pairs = len(METRICS) * len(SUBSET_SIZES)
_t0_overall = _time.time()
_pair_idx = 0
_n_pool = len(points)
print(f'Subset generation: {_n_pairs} (metric, r) pairs over '
      f'{_n_pool} mixed-pool candidates; per-pair combos = '
      f'C({_n_pool}, r). Ledger persists after each pair.')
for metric in METRICS:
    for r in SUBSET_SIZES:
        _pair_idx += 1
        # Skip-if-cached: trust the ledger entry only if every
        # subset.traj under this (metric, r) is also on disk
        # (otherwise the cache is stale / partial).
        _all_present = (
            (metric, r) in subset_index_log
            and all(
                (STEP7_ROOT / metric /
                 f'bin{r:02d}' /
                 ARCH_NAME / LOSS_NAME / solver / 'subset.traj').exists()
                for solver in SOLVERS
            )
        )
        if _all_present:
            print(f'[{_pair_idx:>2d}/{_n_pairs}] metric={metric} r={r:>2d}  '
                  f'CACHED (ledger + all subset.traj present); skip.',
                  flush=True)
            continue
        _ncombo = _math.comb(_n_pool, r)
        _t0 = _time.time()
        print(f'[{_pair_idx:>2d}/{_n_pairs}] metric={metric} r={r:>2d}  '
              f'enumerating C({_n_pool},{r})={_ncombo:,} combos...',
              flush=True)
        _save_dist = _os_smoke.environ.get('STEP7_SAVE_DISTRIBUTIONS', '0') == '1'
        _dist_path = (str(DISTRIBUTIONS_DIR /
                          f'{metric}_r{r:02d}.npz')
                      if _save_dist else None)
        if _save_dist:
            chosen, val, _vals_all, _idx_all = ss.select_subset(
                point_descriptors, edges, h_ref, r=r, metric=metric,
                descriptor_weights=DESCRIPTOR_WEIGHTS,
                progress_desc=f'{metric} r={r}',
                return_all=True,
                distribution_path=_dist_path)
        else:
            chosen, val = ss.select_subset(
                point_descriptors, edges, h_ref, r=r, metric=metric,
                descriptor_weights=DESCRIPTOR_WEIGHTS,
                progress_desc=f'{metric} r={r}')
        _dt = _time.time() - _t0
        _elapsed = _time.time() - _t0_overall
        _eta = (_elapsed / _pair_idx) * (_n_pairs - _pair_idx)
        chosen_points = [points[i] for i in chosen]
        print(f'    -> chosen indices={list(chosen)}  '
              f'metric_value={val:.6e}  dt={_dt:.1f}s  '
              f'elapsed={_elapsed/60:.1f}min  eta={_eta/60:.1f}min',
              flush=True)
        for _tp in chosen_points:
            print(f'         [{_tp.kind:4s}] {_tp.name:24s} '
                  f'species=({", ".join(s.info["name"] for s in _tp.species)})',
                  flush=True)
        # Materialize the chosen points' deduplicated species union
        # as subset.traj for inspection (the ledger is the canonical
        # source of truth for spec reconstruction).
        traj_atoms = species_union_from_points(chosen_points)
        tag = f'bin{r:02d}'
        for solver in SOLVERS:
            spec_dir = (STEP7_ROOT / metric / tag /
                        f'{ARCH_NAME}/{LOSS_NAME}/{solver}')
            spec_dir.mkdir(parents=True, exist_ok=True)
            ase_write(str(spec_dir / 'subset.traj'), traj_atoms)
        subset_index_log[(metric, r)] = {
            'chosen_indices': list(chosen),
            'metric_value': float(val),
            'point_kinds': [_tp.kind for _tp in chosen_points],
            'point_names': [_tp.name for _tp in chosen_points],
            'tag': tag,
        }
        # Persist ledger AFTER each (metric, r) pair so a kill mid-run
        # leaves a self-consistent on-disk record.
        _write_ledger()
        print(f'    [ledger] {len(subset_index_log)} entries persisted '
              f'to {ledger_path.name}', flush=True)
_write_ledger()  # final flush (idempotent if last pair was cached)
n_specs = len(subset_index_log) * len(SOLVERS)
print(f'Wrote {len(subset_index_log)} (metric, r) entries to {ledger_path}')
print(f'Total subset.traj files written: {n_specs}')
if not _SMOKE:
    _expected = len(SUBSET_SIZES) * len(METRICS) * len(SOLVERS)
    assert n_specs == _expected, (
        f'Expected {_expected} specs '
        f'({len(SUBSET_SIZES)} sizes x {len(METRICS)} metrics x '
        f'{len(SOLVERS)} solvers), got {n_specs}')
